In [28]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sandyhanysad/clean-review/cleaned_reviews.csv


In [29]:
import pandas as pd
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)


from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [30]:
df = pd.read_csv("/kaggle/input/datasets/sandyhanysad/clean-review/cleaned_reviews.csv")
print(f'Total rows: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print('\nRating distribution:')
print(df['rating'].value_counts().sort_index())
df.head(3)

Total rows: 326825
Columns: ['userId', 'productId', 'text', 'rating', 'summary']

Rating distribution:
rating
1     25630
2     17253
3     27573
4     51975
5    204394
Name: count, dtype: int64


,userId,productId,text,rating,summary
0,ABXLMWJIXXAIN,B000LQOCH0,This is a confection that has been around a fe...,4,"""Delight"" says it all"
1,A395BORC6FGVXV,B000UA0QIQ,If you are looking for the secret ingredient i...,2,Cough Medicine
2,A21BT40VZCCYT4,B00171APVA,This is a very healthy dog food. Good for thei...,5,Healthy Dog Food


In [31]:
def map_sentiment(rating):
    if rating <= 2:
        return 0  # negative
    elif rating == 3:
        return 1  # neutral
    else:
        return 2  # positive

label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}

df['label'] = df['rating'].apply(map_sentiment)
df['sentiment'] = df['label'].map(label_names)

print('Sentiment distribution:')
print(df['sentiment'].value_counts())

Sentiment distribution:
sentiment
positive    256369
negative     42883
neutral      27573
Name: count, dtype: int64


In [32]:
# Balanced sample: 5000 per class for fine-tuning
SAMPLE_PER_CLASS = 5000

df_sampled = (
    df.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)

print(f'Sampled dataset size: {len(df_sampled)}')
print(df_sampled['sentiment'].value_counts())

Sampled dataset size: 15000
sentiment
negative    5000
neutral     5000
positive    5000
Name: count, dtype: int64


/tmp/ipykernel_57/3810697547.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_CLASS), random_state=42))


In [33]:
train_df, temp_df = train_test_split(
    df_sampled, test_size=0.2, random_state=42, stratify=df_sampled['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 12000 | Val: 1500 | Test: 1500


In [34]:
MODEL_NAME = 'bert-base-uncased'  
MAX_LEN = 128
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f'Loaded tokenizer: {MODEL_NAME}')

Loaded tokenizer: bert-base-uncased


In [35]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }


train_dataset = ReviewDataset(train_df['text'], train_df['label'], tokenizer, MAX_LEN)
val_dataset   = ReviewDataset(val_df['text'],   val_df['label'],   tokenizer, MAX_LEN)
test_dataset  = ReviewDataset(test_df['text'],  test_df['label'],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 750 | Val batches: 94


In [36]:
# Load pretrained BERT with classification head (3 classes)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    output_attentions=False,
    output_hidden_states=False
)
model = model.to(device)
print('Model loaded and moved to device')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to device
Total parameters: 109,484,547
Trainable parameters: 109,484,547


In [37]:
# Fine-tuning hyperparameters
EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f'Total training steps: {total_steps}')

Total training steps: 2250


In [38]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, total_correct = 0, 0

    for batch in tqdm(loader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device).long()

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / len(loader.dataset)

    return avg_loss, accuracy




In [39]:
def eval_epoch(model, loader, device):
    model.eval()
    total_loss, total_correct = 0, 0

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device).long()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / len(loader.dataset)

    return avg_loss, accuracy

In [40]:
history = []
best_val_acc = 0

for epoch in range(1, EPOCHS + 1):
    print(f'\n===== Epoch {epoch}/{EPOCHS} =====')

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = eval_epoch(model, val_loader, device)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}')

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    })

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history
        }, 'best_bert_sentiment.pt')

        print(f'   Best model saved (val_acc={val_acc:.4f})')

print(f'\nBest Validation Accuracy: {best_val_acc:.4f}')


===== Epoch 1/3 =====


Evaluating: 100%|██████████| 94/94 [00:11<00:00,  7.90it/s]


Train Loss: 0.7413 | Train Acc: 0.6573
Val   Loss: 0.5836 | Val   Acc: 0.7613
   Best model saved (val_acc=0.7613)

===== Epoch 2/3 =====


Evaluating: 100%|██████████| 94/94 [00:11<00:00,  7.94it/s]


Train Loss: 0.2697 | Train Acc: 0.9062
Val   Loss: 0.6780 | Val   Acc: 0.7780
   Best model saved (val_acc=0.7780)

Best Validation Accuracy: 0.7780


In [41]:
checkpoint = torch.load('best_bert_sentiment.pt', map_location=device)
model.load_state_dict(checkpoint['model_state'])

model.to(device)
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print('\n=== Test Set Classification Report ===')
print(classification_report(
    all_labels,
    all_preds,
    target_names=['negative', 'neutral', 'positive']
))

print(f'Overall Accuracy: {accuracy_score(all_labels, all_preds):.4f}')

Testing: 100%|██████████| 94/94 [00:11<00:00,  8.03it/s]


=== Test Set Classification Report ===
              precision    recall  f1-score   support

    negative       0.79      0.76      0.78       500
     neutral       0.68      0.73      0.70       500
    positive       0.87      0.84      0.85       500

    accuracy                           0.78      1500
   macro avg       0.78      0.78      0.78      1500
weighted avg       0.78      0.78      0.78      1500

Overall Accuracy: 0.7753


In [42]:
def predict_sentiment(texts, model, tokenizer, device, batch_size=64):
   
    model.eval()
    all_preds = []

    for i in tqdm(range(0, len(texts), batch_size), desc='Inference'):
        batch_texts = texts[i:i+batch_size].tolist()

        encodings = tokenizer(
            batch_texts,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids      = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())

    return all_preds


# Run on full dataset
print(f'Running inference on {len(df)} reviews...')
df['predicted_label'] = predict_sentiment(df['text'], model, tokenizer, device)
df['predicted_sentiment'] = df['predicted_label'].map(label_names)

print('Done!')
print(df['predicted_sentiment'].value_counts())

Running inference on 326825 reviews...


Inference: 100%|██████████| 5107/5107 [39:20<00:00,  2.16it/s]

Done!
predicted_sentiment
positive    219820
neutral      60795
negative     46210
Name: count, dtype: int64


In [45]:
 #Map sentiment labels to numeric scores for averaging
sentiment_score_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['sentiment_score'] = df['predicted_sentiment'].map(sentiment_score_map)

# Aggregate per product
product_agg = df.groupby('productId').agg(
    review_count        = ('predicted_sentiment', 'count'),
    avg_sentiment_score = ('sentiment_score', 'mean'),
    positive_count      = ('predicted_sentiment', lambda x: (x == 'positive').sum()),
    neutral_count       = ('predicted_sentiment', lambda x: (x == 'neutral').sum()),
    negative_count      = ('predicted_sentiment', lambda x: (x == 'negative').sum()),
).reset_index()

# Percentage breakdown
product_agg['positive_pct'] = (product_agg['positive_count'] / product_agg['review_count'] * 100).round(2)
product_agg['neutral_pct']  = (product_agg['neutral_count']  / product_agg['review_count'] * 100).round(2)
product_agg['negative_pct'] = (product_agg['negative_count'] / product_agg['review_count'] * 100).round(2)

# Dominant sentiment label
product_agg['dominant_sentiment'] = product_agg[['positive_count','neutral_count','negative_count']].idxmax(axis=1).str.replace('_count', '')

print(f'Total products: {len(product_agg)}')
product_agg.head(5)

Total products: 45912


,productId,review_count,avg_sentiment_score,positive_count,neutral_count,negative_count,positive_pct,neutral_pct,negative_pct,dominant_sentiment
0,0006641040,7,1.714286,5,2,0,71.43,28.57,0.00,positive
1,2841233731,1,2.000000,1,0,0,100.00,0.00,0.00,positive
2,7310172001,172,1.819767,147,19,6,85.47,11.05,3.49,positive
3,7310172101,172,1.819767,147,19,6,85.47,11.05,3.49,positive
4,7800648702,2,1.000000,0,2,0,0.00,100.00,0.00,neutral


In [48]:
os.makedirs('outputs', exist_ok=True)

# Review-level output
review_output = df[['userId', 'productId', 'rating', 'predicted_sentiment', 'sentiment_score', 'text']].copy()
review_output.rename(columns={'predicted_sentiment': 'sentiment'}, inplace=True)

# Merge with product aggregation
final_output = review_output.merge(
    product_agg[['productId', 'avg_sentiment_score', 'dominant_sentiment',
                 'positive_pct', 'neutral_pct', 'negative_pct', 'review_count']],
    on='productId',
    how='left'
)

# Save
output_path = 'outputs/task2_sentiment.csv'
final_output.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Shape: {final_output.shape}')
final_output.head(3)

Saved: outputs/task2_sentiment.csv
Shape: (326825, 12)


,userId,productId,rating,sentiment,sentiment_score,text,avg_sentiment_score,dominant_sentiment,positive_pct,neutral_pct,negative_pct,review_count
0,ABXLMWJIXXAIN,B000LQOCH0,4,positive,2,This is a confection that has been around a fe...,2.0,positive,100.0,0.0,0.0,1
1,A395BORC6FGVXV,B000UA0QIQ,2,negative,0,If you are looking for the secret ingredient i...,0.0,negative,0.0,0.0,100.0,1
2,A21BT40VZCCYT4,B00171APVA,5,positive,2,This is a very healthy dog food. Good for thei...,2.0,positive,100.0,0.0,0.0,1


In [54]:
import shutil
import os

src = 'outputs/task2_sentiment.csv'
dst = '/kaggle/working/task2_sentiment.csv'

shutil.copy(src, dst)
print("done!")
print(os.path.exists(dst))

done!
True


In [52]:
import os
os.listdir('outputs/')

['task2_sentiment.csv']

In [55]:
from IPython.display import FileLink
FileLink('outputs/task2_sentiment.csv')

/kaggle/working/outputs/task2_sentiment.csv

In [58]:
import os

for f in os.listdir('/kaggle/working'):
    size = os.path.getsize(f'/kaggle/working/{f}')
    print(f'{f} — {size/1e6:.1f} MB')

task2_sentiment.csv — 184.4 MB
best_bert_sentiment.pt — 1314.1 MB
.virtual_documents — 0.0 MB
outputs — 0.0 MB


In [59]:
from IPython.display import FileLink, display

display(FileLink('best_bert_sentiment.pt'))
display(FileLink('task2_sentiment.csv'))

/kaggle/working/best_bert_sentiment.pt

/kaggle/working/task2_sentiment.csv